# CampusDesk AI — RAG Logic Walkthrough

This notebook demonstrates, step by step, the **Retrieval-Augmented Generation (RAG)** logic used in the CampusDesk AI chatbot's PDF-upload feature — separate from the Flask web app, for clarity and grading purposes.

**Pipeline:**
1. Extract text from a document
2. Split text into overlapping chunks
3. Convert chunks + a question into TF-IDF vectors
4. Use cosine similarity to find the most relevant chunks
5. (In the full app) Pass those chunks to Gemini to generate a grounded answer


## 1. Sample document text

In the real app this comes from an uploaded PDF (via `pypdf`). Here we use plain text to keep the demo self-contained.

In [1]:
sample_document = """
Admissions require the following documents: 10th and 12th mark sheets, transfer certificate,
migration certificate, ID proof, passport-size photographs, and category certificate if applicable.

The exam timetable is published on the college notice board and the official website under the
Examinations section, usually 2-3 weeks before exams begin.

Fee refunds require a written request submitted to the accounts office along with the fee receipt.
Refunds are processed within 15-20 working days as per the institution's refund policy.

The library allows students to borrow up to 3 books at a time for a period of 14 days, renewable
once if no one else has requested the book.

Hostel applications are submitted through the student portal along with admission proof. Rooms
are allotted on a first-come, first-served basis, subject to availability.
"""
print(sample_document)


Admissions require the following documents: 10th and 12th mark sheets, transfer certificate,
migration certificate, ID proof, passport-size photographs, and category certificate if applicable.

The exam timetable is published on the college notice board and the official website under the
Examinations section, usually 2-3 weeks before exams begin.

Fee refunds require a written request submitted to the accounts office along with the fee receipt.
Refunds are processed within 15-20 working days as per the institution's refund policy.

The library allows students to borrow up to 3 books at a time for a period of 14 days, renewable
once if no one else has requested the book.

Hostel applications are submitted through the student portal along with admission proof. Rooms
are allotted on a first-come, first-served basis, subject to availability.



## 2. Chunking

We split the document into overlapping word chunks. Overlap prevents important context from being cut off exactly at a chunk boundary.

In [2]:
def chunk_text(text, chunk_size=40, overlap=10):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(sample_document)
for i, c in enumerate(chunks):
    print(f"--- Chunk {i} ---")
    print(c)
    print()

--- Chunk 0 ---
Admissions require the following documents: 10th and 12th mark sheets, transfer certificate, migration certificate, ID proof, passport-size photographs, and category certificate if applicable. The exam timetable is published on the college notice board and the official website under the Examinations

--- Chunk 1 ---
college notice board and the official website under the Examinations section, usually 2-3 weeks before exams begin. Fee refunds require a written request submitted to the accounts office along with the fee receipt. Refunds are processed within 15-20 working days

--- Chunk 2 ---
the fee receipt. Refunds are processed within 15-20 working days as per the institution's refund policy. The library allows students to borrow up to 3 books at a time for a period of 14 days, renewable once if no one

--- Chunk 3 ---
a period of 14 days, renewable once if no one else has requested the book. Hostel applications are submitted through the student portal along with admis

## 3. TF-IDF Vectorization

TF-IDF (Term Frequency – Inverse Document Frequency) converts text into numeric vectors, weighting words by how distinctive they are — common words like "the" get low weight, distinctive words like "refund" get high weight.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_relevant_chunks(question, chunks, top_k=2):
    vectorizer = TfidfVectorizer(stop_words="english")
    all_texts = chunks + [question]
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    question_vector = tfidf_matrix[-1]
    chunk_vectors = tfidf_matrix[:-1]

    similarities = cosine_similarity(question_vector, chunk_vectors)[0]
    top_indices = similarities.argsort()[::-1][:top_k]

    return [(chunks[i], similarities[i]) for i in top_indices]

## 4. Test retrieval with a few sample questions

Notice how the similarity score is highest for the chunk that actually contains the relevant information.

In [4]:
test_questions = [
    "How do I check my exam timetable?",
    "What is the fee refund process?",
    "How many books can I borrow from the library?",
]

for q in test_questions:
    print(f"Question: {q}")
    results = retrieve_relevant_chunks(q, chunks)
    for chunk, score in results:
        print(f"  [score={score:.3f}] {chunk[:100]}...")
    print()

Question: How do I check my exam timetable?
  [score=0.156] Admissions require the following documents: 10th and 12th mark sheets, transfer certificate, migrati...
  [score=0.000] on a first-come, first-served basis, subject to availability....

Question: What is the fee refund process?
  [score=0.190] the fee receipt. Refunds are processed within 15-20 working days as per the institution's refund pol...
  [score=0.137] college notice board and the official website under the Examinations section, usually 2-3 weeks befo...

Question: How many books can I borrow from the library?
  [score=0.348] the fee receipt. Refunds are processed within 15-20 working days as per the institution's refund pol...
  [score=0.000] on a first-come, first-served basis, subject to availability....



## 5. Generation step (conceptual)

In the full Flask app, the top-matching chunk(s) above are inserted into a system prompt sent to the **Gemini API**, which generates the final natural-language answer grounded in that retrieved content. This is the "generation" half of Retrieval-Augmented Generation — shown here conceptually since it requires a live API key.

In [5]:
# Conceptual illustration (not executed here — requires GEMINI_API_KEY)
#
# import google.generativeai as genai
# genai.configure(api_key=os.environ["GEMINI_API_KEY"])
# model = genai.GenerativeModel("gemini-3.5-flash", system_instruction=f"Answer using: {retrieved_chunk}")
# response = model.generate_content(question)
# print(response.text)

print("See app.py in the main project for the live implementation.")

See app.py in the main project for the live implementation.


## 6. Observations

- TF-IDF retrieval works well for prose/paragraph content, correctly matching questions to the right chunk.
- A known limitation (documented in the project report) is that **dense tabular data** (e.g., credit tables) does not extract or match cleanly with this approach, since text extraction flattens table structure.
- This lightweight approach was chosen over heavier embedding-based retrieval for reliability and speed on limited hardware/time, which is a valid engineering trade-off for a project of this scope.